#Problem Statement : Hospital Patient Data Analysis

Context:
A hospital maintains patient records including admission details, department, diagnosis, doctor, and bill amount. You have two datasets: one with patient info and another with billing details. Some patients have blank bill amounts, and there are multiple rows for the same patient due to follow-ups.


## Tasks:

In [2]:
# Importing packages
import pandas as pd
import numpy as np
import warnings
warnings.filterwarnings('ignore')
import matplotlib.pyplot as plt
import seaborn as sns

1.	Load the patient dataset and show summary with info().

In [44]:
df = pd.read_csv('Patient_Data.csv')
df2=pd.read_csv('Billing_Data.csv')

In [9]:
df.head()

,PatientID,Name,Department,Doctor,BillAmount,ReceptionistID,CheckInTime
0,101,Alice,Cardiology,Dr. Smith,5000.0,1,2023-01-10 09:00
1,102,Bob,Neurology,Dr. John,NaN,2,2023-01-11 10:30
2,103,Charlie,Orthopedics,Dr. Lee,7500.0,1,2023-01-12 11:00
3,104,David,Cardiology,Dr. Smith,6200.0,3,2023-01-13 12:00
4,105,Eva,Dermatology,Dr. Rose,NaN,2,2023-01-14 08:45


In [10]:
df.shape

(6, 7)

In [11]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 6 entries, 0 to 5
Data columns (total 7 columns):
 #   Column          Non-Null Count  Dtype  
---  ------          --------------  -----  
 0   PatientID       6 non-null      int64  
 1   Name            6 non-null      object 
 2   Department      6 non-null      object 
 3   Doctor          6 non-null      object 
 4   BillAmount      4 non-null      float64
 5   ReceptionistID  6 non-null      int64  
 6   CheckInTime     6 non-null      object 
dtypes: float64(1), int64(2), object(4)
memory usage: 468.0+ bytes


2.	Select only the columns relevant for billing: ['PatientID', 'Department', 'Doctor', 'BillAmount'].

In [16]:
df_related =df[['PatientID','Department','Doctor','BillAmount']]

In [18]:
df_related.head()

,PatientID,Department,Doctor,BillAmount
0,101,Cardiology,Dr. Smith,5000.0
1,102,Neurology,Dr. John,NaN
2,103,Orthopedics,Dr. Lee,7500.0
3,104,Cardiology,Dr. Smith,6200.0
4,105,Dermatology,Dr. Rose,NaN


In [19]:
df_related.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 6 entries, 0 to 5
Data columns (total 4 columns):
 #   Column      Non-Null Count  Dtype  
---  ------      --------------  -----  
 0   PatientID   6 non-null      int64  
 1   Department  6 non-null      object 
 2   Doctor      6 non-null      object 
 3   BillAmount  4 non-null      float64
dtypes: float64(1), int64(1), object(2)
memory usage: 324.0+ bytes


3.	Drop administrative columns like ['ReceptionistID', 'CheckInTime'].


In [22]:
df_dropped = df.drop(['ReceptionistID','CheckInTime'],axis=1)

In [24]:
df_dropped.head()

,PatientID,Name,Department,Doctor,BillAmount
0,101,Alice,Cardiology,Dr. Smith,5000.0
1,102,Bob,Neurology,Dr. John,NaN
2,103,Charlie,Orthopedics,Dr. Lee,7500.0
3,104,David,Cardiology,Dr. Smith,6200.0
4,105,Eva,Dermatology,Dr. Rose,NaN


In [25]:
df_dropped.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 6 entries, 0 to 5
Data columns (total 5 columns):
 #   Column      Non-Null Count  Dtype  
---  ------      --------------  -----  
 0   PatientID   6 non-null      int64  
 1   Name        6 non-null      object 
 2   Department  6 non-null      object 
 3   Doctor      6 non-null      object 
 4   BillAmount  4 non-null      float64
dtypes: float64(1), int64(1), object(3)
memory usage: 372.0+ bytes


4.	Use groupby to find total bill amount per department.

In [27]:
df_groupby_billamount = df.groupby('Department')['BillAmount'].sum()

In [28]:
df_groupby_billamount

Department
Cardiology     16200.0
Dermatology        0.0
Neurology          0.0
Orthopedics     7500.0
Name: BillAmount, dtype: float64

5.	Remove duplicate patient records based on PatientID.

In [33]:
df.duplicated().sum()

np.int64(1)

In [34]:
df[df.duplicated()]

,PatientID,Name,Department,Doctor,BillAmount,ReceptionistID,CheckInTime
5,101,Alice,Cardiology,Dr. Smith,5000.0,1,2023-01-10 09:00


In [37]:
# dropping duplicates based on the PatientID
df_no_duplicates = df.drop_duplicates(subset='PatientID',keep='last')
df_no_duplicates.head()


,PatientID,Name,Department,Doctor,BillAmount,ReceptionistID,CheckInTime
1,102,Bob,Neurology,Dr. John,NaN,2,2023-01-11 10:30
2,103,Charlie,Orthopedics,Dr. Lee,7500.0,1,2023-01-12 11:00
3,104,David,Cardiology,Dr. Smith,6200.0,3,2023-01-13 12:00
4,105,Eva,Dermatology,Dr. Rose,NaN,2,2023-01-14 08:45
5,101,Alice,Cardiology,Dr. Smith,5000.0,1,2023-01-10 09:00


In [38]:
df_no_duplicates.duplicated().sum()

np.int64(0)

6.	Fill missing BillAmount values with the mean bill amount.

In [39]:
df.isnull().sum()

PatientID         0
Name              0
Department        0
Doctor            0
BillAmount        2
ReceptionistID    0
CheckInTime       0
dtype: int64

In [41]:
## Filling missing values with mean
df_fillna_mean = df.fillna(df['BillAmount'].mean())

In [43]:
df_fillna_mean.isnull().sum()

PatientID         0
Name              0
Department        0
Doctor            0
BillAmount        0
ReceptionistID    0
CheckInTime       0
dtype: int64

7.	Merge the billing dataset with patient dataset on PatientID.

In [45]:
df2.head()

,PatientID,InsuranceCovered,FinalAmount
0,101,2000,3000
1,102,1500,3500
2,103,2500,5000
3,104,3000,3200
4,105,1000,4000


In [49]:
# merging datasets df,df2 based on Patientid
df_merged = pd.merge(df,df2,on='PatientID')

In [50]:
df_merged.head()

,PatientID,Name,Department,Doctor,BillAmount,ReceptionistID,CheckInTime,InsuranceCovered,FinalAmount
0,101,Alice,Cardiology,Dr. Smith,5000.0,1,2023-01-10 09:00,2000,3000
1,102,Bob,Neurology,Dr. John,NaN,2,2023-01-11 10:30,1500,3500
2,103,Charlie,Orthopedics,Dr. Lee,7500.0,1,2023-01-12 11:00,2500,5000
3,104,David,Cardiology,Dr. Smith,6200.0,3,2023-01-13 12:00,3000,3200
4,105,Eva,Dermatology,Dr. Rose,NaN,2,2023-01-14 08:45,1000,4000


In [51]:
df_merged.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 6 entries, 0 to 5
Data columns (total 9 columns):
 #   Column            Non-Null Count  Dtype  
---  ------            --------------  -----  
 0   PatientID         6 non-null      int64  
 1   Name              6 non-null      object 
 2   Department        6 non-null      object 
 3   Doctor            6 non-null      object 
 4   BillAmount        4 non-null      float64
 5   ReceptionistID    6 non-null      int64  
 6   CheckInTime       6 non-null      object 
 7   InsuranceCovered  6 non-null      int64  
 8   FinalAmount       6 non-null      int64  
dtypes: float64(1), int64(4), object(4)
memory usage: 564.0+ bytes


8.	Concatenate an additional DataFrame that contains new patients for the current week (row-wise).

In [56]:
df_concat = pd.concat([df,df2],ignore_index=True)

In [57]:
df_concat

,PatientID,Name,Department,Doctor,BillAmount,ReceptionistID,CheckInTime,InsuranceCovered,FinalAmount
0,101,Alice,Cardiology,Dr. Smith,5000.0,1.0,2023-01-10 09:00,NaN,NaN
1,102,Bob,Neurology,Dr. John,NaN,2.0,2023-01-11 10:30,NaN,NaN
2,103,Charlie,Orthopedics,Dr. Lee,7500.0,1.0,2023-01-12 11:00,NaN,NaN
3,104,David,Cardiology,Dr. Smith,6200.0,3.0,2023-01-13 12:00,NaN,NaN
4,105,Eva,Dermatology,Dr. Rose,NaN,2.0,2023-01-14 08:45,NaN,NaN
5,101,Alice,Cardiology,Dr. Smith,5000.0,1.0,2023-01-10 09:00,NaN,NaN
6,101,NaN,NaN,NaN,NaN,NaN,NaN,2000.0,3000.0
7,102,NaN,NaN,NaN,NaN,NaN,NaN,1500.0,3500.0
8,103,NaN,NaN,NaN,NaN,NaN,NaN,2500.0,5000.0
9,104,NaN,NaN,NaN,NaN,NaN,NaN,3000.0,3200.0


In [58]:
df_concat.shape

(11, 9)

9.	Concatenate new billing category columns like ['InsuranceCovered', 'FinalAmount'] (column-wise).

In [60]:
# declaring new data frame with new columns
new_billing_cols = pd.DataFrame({
    'InsuranceCovered':[True, False, True,True,False,True,True],
    'FinalAmount':[1200, 900, 1500,2000,1000,500,700]})
    

In [61]:
new_billing_cols

,InsuranceCovered,FinalAmount
0,True,1200
1,False,900
2,True,1500
3,True,2000
4,False,1000
5,True,500
6,True,700


In [64]:
df_new_added_columns = pd.concat([df_concat,new_billing_cols],axis=1)

In [68]:
df_new_added_columns.head()

,PatientID,Name,Department,Doctor,BillAmount,ReceptionistID,CheckInTime,InsuranceCovered,FinalAmount,InsuranceCovered,FinalAmount
0,101,Alice,Cardiology,Dr. Smith,5000.0,1.0,2023-01-10 09:00,NaN,NaN,True,1200.0
1,102,Bob,Neurology,Dr. John,NaN,2.0,2023-01-11 10:30,NaN,NaN,False,900.0
2,103,Charlie,Orthopedics,Dr. Lee,7500.0,1.0,2023-01-12 11:00,NaN,NaN,True,1500.0
3,104,David,Cardiology,Dr. Smith,6200.0,3.0,2023-01-13 12:00,NaN,NaN,True,2000.0
4,105,Eva,Dermatology,Dr. Rose,NaN,2.0,2023-01-14 08:45,NaN,NaN,False,1000.0


In [69]:
df_new_added_columns.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 11 entries, 0 to 10
Data columns (total 11 columns):
 #   Column            Non-Null Count  Dtype  
---  ------            --------------  -----  
 0   PatientID         11 non-null     int64  
 1   Name              6 non-null      object 
 2   Department        6 non-null      object 
 3   Doctor            6 non-null      object 
 4   BillAmount        4 non-null      float64
 5   ReceptionistID    6 non-null      float64
 6   CheckInTime       6 non-null      object 
 7   InsuranceCovered  5 non-null      float64
 8   FinalAmount       5 non-null      float64
 9   InsuranceCovered  7 non-null      object 
 10  FinalAmount       7 non-null      float64
dtypes: float64(5), int64(1), object(5)
memory usage: 1.1+ KB
